# 06 - Defense Evaluation

Evaluate three defense strategies against adversarial attacks on the IDS:

| Defense | Approach | Module |
|---------|----------|--------|
| **Adversarial Training** | Train with FGSM + PGD augmented batches | `AdversarialTrainer` |
| **Input Transformation** | Bit-depth reduction, Gaussian smoothing | `input_transformation` |
| **Ensemble Voting** | Combine predictions from diverse models | `EnsembleDefense` |

**Sections:**
1. Setup: data, baseline model
2. Adversarial training
3. Input transformation defense
4. Ensemble defense
5. Compare robust vs non-robust models

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader as TorchDataLoader, TensorDataset

from src.preprocessing.data_loader import DataLoader
from src.preprocessing.preprocessor import DataPreprocessor
from src.adversarial_attacks.fgsm import fgsm_attack
from src.adversarial_attacks.pgd import pgd_attack
from src.adversarial_attacks.attack_utils import PyTorchDNN, evaluate_attack
from src.tier3_adversarial_defense.adversarial_training import AdversarialTrainer
from src.tier3_adversarial_defense.input_transformation import (
    bit_depth_reduction, gaussian_smoothing, feature_squeezing
)
from src.tier3_adversarial_defense.ensemble_defense import EnsembleDefense, ModelWrapper
from src.utils.config import load_config

%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 5)
sns.set_style('whitegrid')

print('Imports loaded.')

## 1. Setup: Data and Baseline Model

In [ ]:
config = load_config('../config/config.yaml')
loader = DataLoader()

try:
    df = loader.load(config)
except FileNotFoundError:
    print('Using synthetic data.')
    df = loader.generate_synthetic(n_samples=5000)

preprocessor = DataPreprocessor(config)
data = preprocessor.run_pipeline(df, label_type='multiclass')

n_features = data['n_features']
n_classes = data['n_classes']
print(f'Features: {n_features}  Classes: {n_classes}')

In [ ]:
# Train a baseline (non-robust) model
baseline_model = PyTorchDNN(n_features, n_classes)
optimizer = torch.optim.Adam(baseline_model.parameters(), lr=0.001)

X_train_t = torch.FloatTensor(data['X_train'])
y_train_t = torch.LongTensor(data['y_train'])

print('Training baseline (non-robust) model...')
baseline_model.train()
for epoch in range(15):
    total_loss = 0
    for i in range(0, len(X_train_t), 256):
        xb = X_train_t[i:i+256]
        yb = y_train_t[i:i+256]
        optimizer.zero_grad()
        loss = F.cross_entropy(baseline_model(xb), yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

# Evaluate
baseline_model.eval()
X_test_t = torch.FloatTensor(data['X_test'])
y_test_t = torch.LongTensor(data['y_test'])
with torch.no_grad():
    baseline_clean_acc = (baseline_model(X_test_t).argmax(1).numpy() == data['y_test']).mean()
print(f'Baseline clean accuracy: {baseline_clean_acc:.4f}')

In [ ]:
# Generate adversarial test set for evaluation
N_ATK = min(500, len(data['X_test']))
x_test_sub = torch.FloatTensor(data['X_test'][:N_ATK])
y_test_sub = torch.LongTensor(data['y_test'][:N_ATK])

eps = config['adversarial_attacks']['fgsm']['epsilon']

x_adv_fgsm = fgsm_attack(baseline_model, x_test_sub, y_test_sub, epsilon=eps)
x_adv_pgd = pgd_attack(baseline_model, x_test_sub, y_test_sub,
                        epsilon=eps, alpha=eps/4, num_iterations=10)

baseline_fgsm = evaluate_attack(baseline_model, x_test_sub, x_adv_fgsm, y_test_sub)
baseline_pgd = evaluate_attack(baseline_model, x_test_sub, x_adv_pgd, y_test_sub)

print(f'Baseline under FGSM: {baseline_fgsm["attack_success_rate"]:.2f}% fooled')
print(f'Baseline under PGD : {baseline_pgd["attack_success_rate"]:.2f}% fooled')

## 2. Adversarial Training

Train the model on a mix of clean + adversarial examples to improve robustness.
Each batch: 40% clean, 30% FGSM, 30% PGD.

In [ ]:
# Create a new model for adversarial training
robust_model = PyTorchDNN(n_features, n_classes)

adv_trainer = AdversarialTrainer(robust_model, config)

train_dataset = TensorDataset(X_train_t, y_train_t)
val_dataset = TensorDataset(
    torch.FloatTensor(data['X_val']),
    torch.LongTensor(data['y_val'])
)

train_loader = TorchDataLoader(train_dataset, batch_size=128, shuffle=True)
val_loader = TorchDataLoader(val_dataset, batch_size=128)

ADV_EPOCHS = min(config['adversarial_training']['epochs'], 10)
print(f'Adversarial training for {ADV_EPOCHS} epochs...')
best_val_acc = adv_trainer.train(train_loader, val_loader, epochs=ADV_EPOCHS)
print(f'\nBest validation accuracy: {best_val_acc:.4f}')

In [ ]:
# Evaluate robust model against the same attacks
robust_model.eval()
with torch.no_grad():
    robust_clean_acc = (robust_model(X_test_t).argmax(1).numpy() == data['y_test']).mean()

x_adv_fgsm_r = fgsm_attack(robust_model, x_test_sub, y_test_sub, epsilon=eps)
x_adv_pgd_r = pgd_attack(robust_model, x_test_sub, y_test_sub,
                          epsilon=eps, alpha=eps/4, num_iterations=10)

robust_fgsm = evaluate_attack(robust_model, x_test_sub, x_adv_fgsm_r, y_test_sub)
robust_pgd = evaluate_attack(robust_model, x_test_sub, x_adv_pgd_r, y_test_sub)

print(f'Robust model clean accuracy: {robust_clean_acc:.4f}')
print(f'Robust under FGSM: {robust_fgsm["attack_success_rate"]:.2f}% fooled')
print(f'Robust under PGD : {robust_pgd["attack_success_rate"]:.2f}% fooled')

## 3. Input Transformation Defense

Apply input transformations to adversarial samples before feeding them to the model.
This can disrupt precise perturbations without significantly harming clean accuracy.

In [ ]:
# Bit depth reduction
x_fgsm_bdr = bit_depth_reduction(x_adv_fgsm.numpy(), depth=4)
x_pgd_bdr = bit_depth_reduction(x_adv_pgd.numpy(), depth=4)

# Gaussian smoothing
x_fgsm_gs = gaussian_smoothing(x_adv_fgsm.numpy(), sigma=0.05)
x_pgd_gs = gaussian_smoothing(x_adv_pgd.numpy(), sigma=0.05)

# Feature squeezing
x_fgsm_fs = feature_squeezing(x_adv_fgsm.numpy(), bit_depth=4)
x_pgd_fs = feature_squeezing(x_adv_pgd.numpy(), bit_depth=4)

print('Input transformations applied.')

In [ ]:
# Evaluate baseline model on transformed adversarial inputs
def get_acc(model, x, y):
    model.eval()
    x_t = torch.FloatTensor(x) if not isinstance(x, torch.Tensor) else x
    with torch.no_grad():
        preds = model(x_t).argmax(dim=1).numpy()
    return (preds == y.numpy() if isinstance(y, torch.Tensor) else preds == y).mean()

transform_results = []

# Clean on clean
clean_on_clean = get_acc(baseline_model, x_test_sub, y_test_sub)

# No defense
fgsm_no_def = get_acc(baseline_model, x_adv_fgsm, y_test_sub)
pgd_no_def = get_acc(baseline_model, x_adv_pgd, y_test_sub)

# Bit depth reduction
fgsm_bdr = get_acc(baseline_model, x_fgsm_bdr, y_test_sub)
pgd_bdr = get_acc(baseline_model, x_pgd_bdr, y_test_sub)

# Gaussian smoothing
fgsm_gs = get_acc(baseline_model, x_fgsm_gs, y_test_sub)
pgd_gs = get_acc(baseline_model, x_pgd_gs, y_test_sub)

# Feature squeezing
fgsm_fs = get_acc(baseline_model, x_fgsm_fs, y_test_sub)
pgd_fs = get_acc(baseline_model, x_pgd_fs, y_test_sub)

transform_df = pd.DataFrame([
    {'Defense': 'No Defense', 'Clean Acc': clean_on_clean, 'FGSM Acc': fgsm_no_def, 'PGD Acc': pgd_no_def},
    {'Defense': 'Bit-Depth Reduction', 'Clean Acc': get_acc(baseline_model, bit_depth_reduction(x_test_sub.numpy()), y_test_sub), 'FGSM Acc': fgsm_bdr, 'PGD Acc': pgd_bdr},
    {'Defense': 'Gaussian Smoothing', 'Clean Acc': get_acc(baseline_model, gaussian_smoothing(x_test_sub.numpy(), 0.05), y_test_sub), 'FGSM Acc': fgsm_gs, 'PGD Acc': pgd_gs},
    {'Defense': 'Feature Squeezing', 'Clean Acc': get_acc(baseline_model, feature_squeezing(x_test_sub.numpy()), y_test_sub), 'FGSM Acc': fgsm_fs, 'PGD Acc': pgd_fs},
])

print('Input Transformation Results:')
print(transform_df.round(4).to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
x_pos = np.arange(len(transform_df))
w = 0.25
ax.bar(x_pos - w, transform_df['Clean Acc'], w, label='Clean', color='#2ecc71')
ax.bar(x_pos, transform_df['FGSM Acc'], w, label='FGSM', color='#e74c3c')
ax.bar(x_pos + w, transform_df['PGD Acc'], w, label='PGD', color='#3498db')
ax.set_xticks(x_pos)
ax.set_xticklabels(transform_df['Defense'], rotation=15, ha='right')
ax.set_ylabel('Accuracy')
ax.set_title('Input Transformation Defense Comparison')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')
ax.set_ylim(0, 1.1)
plt.tight_layout()
plt.show()

## 4. Ensemble Defense

Combine predictions from multiple diverse models. Different architectures make
different errors on adversarial inputs, so ensemble voting improves robustness.

In [ ]:
# Train two additional models with different architectures/seeds
model_a = PyTorchDNN(n_features, n_classes)
model_b = PyTorchDNN(n_features, n_classes)

for name, model in [('Model A', model_a), ('Model B', model_b)]:
    opt = torch.optim.Adam(model.parameters(), lr=0.001)
    model.train()
    # Shuffle differently for diversity
    perm = torch.randperm(len(X_train_t))
    for epoch in range(10):
        for i in range(0, len(X_train_t), 256):
            idx = perm[i:i+256]
            opt.zero_grad()
            loss = F.cross_entropy(model(X_train_t[idx]), y_train_t[idx])
            loss.backward()
            opt.step()
    model.eval()
    with torch.no_grad():
        acc = (model(X_test_t).argmax(1).numpy() == data['y_test']).mean()
    print(f'{name} clean accuracy: {acc:.4f}')

print('Baseline model clean accuracy:', f'{baseline_clean_acc:.4f}')

In [ ]:
# Create ensemble
wrapper_base = ModelWrapper(baseline_model, framework='pytorch')
wrapper_a = ModelWrapper(model_a, framework='pytorch')
wrapper_b = ModelWrapper(model_b, framework='pytorch')

ensemble = EnsembleDefense([wrapper_base, wrapper_a, wrapper_b])

# Evaluate on clean data
ens_clean = ensemble.predict(x_test_sub.numpy())
ens_clean_acc = (ens_clean['predictions'] == y_test_sub.numpy()).mean()

# Evaluate on adversarial data
ens_fgsm = ensemble.predict(x_adv_fgsm.numpy())
ens_fgsm_acc = (ens_fgsm['predictions'] == y_test_sub.numpy()).mean()

ens_pgd = ensemble.predict(x_adv_pgd.numpy())
ens_pgd_acc = (ens_pgd['predictions'] == y_test_sub.numpy()).mean()

print(f'Ensemble clean accuracy : {ens_clean_acc:.4f}')
print(f'Ensemble FGSM accuracy  : {ens_fgsm_acc:.4f}')
print(f'Ensemble PGD accuracy   : {ens_pgd_acc:.4f}')
print(f'Avg agreement (clean)   : {ens_clean["agreement"].mean():.4f}')
print(f'Avg agreement (FGSM)    : {ens_fgsm["agreement"].mean():.4f}')

## 5. Compare All Defenses: Robust vs Non-Robust

In [ ]:
# Robust model accuracies on adversarial data
robust_fgsm_acc = get_acc(robust_model, x_adv_fgsm_r, y_test_sub)
robust_pgd_acc = get_acc(robust_model, x_adv_pgd_r, y_test_sub)

summary = pd.DataFrame([
    {'System': 'Baseline (no defense)', 'Clean Acc': baseline_clean_acc,
     'FGSM Acc': fgsm_no_def, 'PGD Acc': pgd_no_def},
    {'System': 'Adversarial Training', 'Clean Acc': robust_clean_acc,
     'FGSM Acc': robust_fgsm_acc, 'PGD Acc': robust_pgd_acc},
    {'System': 'Input Transform (BDR)', 'Clean Acc': get_acc(baseline_model, bit_depth_reduction(x_test_sub.numpy()), y_test_sub),
     'FGSM Acc': fgsm_bdr, 'PGD Acc': pgd_bdr},
    {'System': 'Ensemble (3 models)', 'Clean Acc': ens_clean_acc,
     'FGSM Acc': ens_fgsm_acc, 'PGD Acc': ens_pgd_acc},
])

print('=== Defense Comparison ===')
print(summary.round(4).to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
x_pos = np.arange(len(summary))
w = 0.25

bars1 = ax.bar(x_pos - w, summary['Clean Acc'], w, label='Clean', color='#2ecc71')
bars2 = ax.bar(x_pos, summary['FGSM Acc'], w, label='Under FGSM', color='#e74c3c')
bars3 = ax.bar(x_pos + w, summary['PGD Acc'], w, label='Under PGD', color='#3498db')

ax.set_xticks(x_pos)
ax.set_xticklabels(summary['System'], rotation=15, ha='right')
ax.set_ylabel('Accuracy')
ax.set_title('Defense Strategy Comparison: Clean vs Adversarial Accuracy')
ax.legend(loc='upper right')
ax.grid(True, alpha=0.3, axis='y')
ax.set_ylim(0, 1.15)

# Annotate bars
for bars in [bars1, bars2, bars3]:
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2, h + 0.01,
                f'{h:.2f}', ha='center', fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
# Accuracy drop comparison
summary['FGSM Drop'] = summary['Clean Acc'] - summary['FGSM Acc']
summary['PGD Drop'] = summary['Clean Acc'] - summary['PGD Acc']

fig, ax = plt.subplots(figsize=(10, 5))
x_pos = np.arange(len(summary))
w = 0.35
ax.bar(x_pos - w/2, summary['FGSM Drop'], w, label='FGSM Drop', color='#e74c3c', alpha=0.8)
ax.bar(x_pos + w/2, summary['PGD Drop'], w, label='PGD Drop', color='#3498db', alpha=0.8)
ax.set_xticks(x_pos)
ax.set_xticklabels(summary['System'], rotation=15, ha='right')
ax.set_ylabel('Accuracy Drop')
ax.set_title('Accuracy Drop Under Attack (Lower is Better)')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

## Summary

Key findings:

- **Baseline model** is highly vulnerable -- accuracy drops significantly under both FGSM and PGD.
- **Adversarial training** provides the strongest defense with the least accuracy drop under attack.
- **Input transformations** (bit-depth reduction, Gaussian smoothing) can partially mitigate attacks but may reduce clean accuracy.
- **Ensemble voting** improves robustness by averaging out individual model vulnerabilities.
- The best defense strategy combines adversarial training with ensemble voting.

Next: `07_final_evaluation.ipynb` -- full system evaluation with all metrics.